# The Cache Is Bigger Than the Model

**Topic:** KV cache and attention variants &nbsp;·&nbsp; **Difficulty:** Medium

At long context the KV cache, not the weights, is what fills the GPU. A 70B model
in fp16 is 140 GB of weights, and its cache at 128k tokens and batch 8 can be
larger still, which is why most serving stacks since 2023 have focused on
shrinking that number.

Compute the cache for four attention variants and find the batch size each one
allows on a fixed card.

## The data (created in the next cell)

- 80 layers, 64 query heads, head dim 128, fp16 (2 bytes).
- Variants: MHA (64 KV heads), MQA (1), GQA-8 (8), MLA (a shared latent of 512
  per token per layer, with DeepSeek-V2's 64-number position key left out).
- `contexts = [4096, 32768, 131072]`, GPU memory 80 GiB with 20 GiB reserved for
  activations, the weights assumed to sit on other cards.

## Your task

With **NumPy only**:

1. Cache bytes per token, totalled over all 80 layers, is
   `2 * n_kv_heads * head_dim * dtype_bytes * n_layers` for the first three (the 2
   is keys plus values). For MLA it is `latent_dim * dtype_bytes * n_layers`
   (one compressed vector, no separate K and V).
2. `cache_gb` `(4, 3)`, cache size in GiB for batch 1 at each context.
3. `max_batch` `(4, 3)`, the largest integer batch fitting in the 60 GiB
   available.
4. Set `answers` to a dict with `"bytes_per_token"` (4 values, summed over all 80
   layers), `"cache_gb"` `(4, 3)` and `"max_batch"` `(4, 3)`, all in the
   order MHA, GQA-8, MQA, MLA, and `"mha_over_mla"`, the
   ratio of MHA's cache to MLA's.

## Evaluation

Compared to the reference at a relative tolerance of 1e-6. The evaluation cell
checks the variants are ordered MHA > GQA-8 > MQA and that MLA's cache is smaller
than MHA's.

In [ ]:
import numpy as np

n_layers, n_q_heads, head_dim, dtype_bytes = 80, 64, 128, 2
kv_heads = {"MHA": 64, "MQA": 1, "GQA8": 8}
mla_latent = 512
contexts = [4096, 32768, 131072]
gpu_bytes = 80 * 1024 ** 3
reserved = 20 * 1024 ** 3
print(f"{n_layers} layers, {n_q_heads} query heads, head dim {head_dim}, fp16")
print(f"available for cache: {(gpu_bytes - reserved) / 1024 ** 3:.0f} GiB")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the constants defined above to compute the cache sizes, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("bytes_per_token", [4], 3072000.0, -310739.94592095964, 1e-06),
    ("cache_gb", [4, 3], 480.46875, -425.4521121514211, 1e-06),
    ("max_batch", [4, 3], 727.0, -283.72980180275255, 1e-06),
    ("mha_over_mla", [], 32.0, -5.358016474063973, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "bytes_per_token" in answers:
    _p = np.asarray(answers["bytes_per_token"], dtype=float)
    _order = _p[0] > _p[1] > _p[2] and _p[3] < _p[0]
    print(f"  {'MHA > GQA > MQA':<24} {'ok' if _order else 'wrong'}  "
          f"(bytes/token {_p.astype(int).tolist()})")
    print(f"  {'MHA/MLA ratio':<24} {answers['mha_over_mla']:.1f}x")
    _ok = _ok and _order

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

MQA shares a single key/value head across all 64 query heads, a 64x saving that
costs measurable quality, which is why GQA's 8 groups became a widely used
compromise. MLA goes further by caching one low-rank latent per token and
reconstructing keys and values on the fly, trading a little compute for a much
smaller cache. The batch numbers are the reason any of this matters: throughput
is roughly proportional to batch size, so a 4x smaller cache is close to 4x the
tokens per second per GPU.

```python
import numpy as np

order = ["MHA", "GQA8", "MQA", "MLA"]
per_token = []
for name in order:
    if name == "MLA":
        per_token.append(mla_latent * dtype_bytes * n_layers)
    else:
        per_token.append(2 * kv_heads[name] * head_dim * dtype_bytes * n_layers)
per_token = np.array(per_token, dtype=float)

ctx = np.array(contexts, dtype=float)
cache_bytes = per_token[:, None] * ctx[None, :]
cache_gb = cache_bytes / 1024 ** 3

avail = float(gpu_bytes - reserved)
max_batch = np.floor(avail / cache_bytes)

answers = {
    "bytes_per_token": per_token,
    "cache_gb": cache_gb,
    "max_batch": max_batch,
    "mha_over_mla": float(per_token[0] / per_token[3]),
}
```